# External Validation and Independence Audit

## Scientific objective

Perform locked external validation for the four Tox21 stress-response endpoints using the official NCATS Tox21 Challenge final-evaluation structures and labels:

- `SR-ARE`
- `SR-ATAD5`
- `SR-MMP`
- `SR-p53`

The workflow preserves the downloaded raw files, prepares endpoint-specific datasets, excludes exact development-set molecule overlaps from strict external metrics, applies the already trained and calibrated endpoint bundles without refitting, reports discrimination/calibration/uncertainty/applicability-domain results, and selects reproducible molecule case studies.

## Inputs

- `data/external/tox21_challenge_final/raw/tox21_10k_challenge_scoresmiles.txt`
- `data/external/tox21_challenge_final/raw/tox21_10k_challenge_score.txt`
- `data/processed/endpoint_records.csv`
- `models/calibrated/*.joblib`

## Scientific safeguards

- External labels are never used for training, model selection, calibration, or threshold tuning.
- Exact standardized-SMILES or InChIKey overlaps are excluded from the strict external analysis.
- Scaffold overlap is retained as a reported stratification variable, not silently removed.
- Results are reported for all strict evaluable molecules and for the decision-supported subset.
- Coverage and abstention rate are reported to avoid selective-performance claims.
- hERG and Ames are not externally validated by this dataset.


In [1]:
import os

for variable in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "BLIS_NUM_THREADS",
]:
    os.environ.setdefault(variable, "1")

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        "Run this notebook from the repository root or notebooks directory"
    )
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)

print(
    {
        "root": str(ROOT),
        "profile": PROFILE,
        "seed": SEED,
        "bootstrap_iterations": PROFILE_CONFIG["bootstrap_iterations"],
    }
)


{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723, 'bootstrap_iterations': 2000}


## 1. Prepare and provenance-lock the official NCATS files

This cell parses the tab-delimited structure and score files, joins them by `Sample ID`, converts `x` to missing labels, standardizes structures with the project's existing chemistry pipeline, writes endpoint-specific processed tables, creates `data/external/external_registry.csv`, and records SHA-256 checksums. The raw files are not modified.


In [2]:
from toxicity_screening.external_validation import (
    EXTERNAL_ENDPOINTS,
    prepare_tox21_external_validation,
)

raw_paths = [
    ROOT
    / "data"
    / "external"
    / "tox21_challenge_final"
    / "raw"
    / "tox21_10k_challenge_scoresmiles.txt",
    ROOT
    / "data"
    / "external"
    / "tox21_challenge_final"
    / "raw"
    / "tox21_10k_challenge_score.txt",
]
missing_raw = [str(path) for path in raw_paths if not path.exists()]
if missing_raw:
    raise FileNotFoundError(
        "Missing official NCATS raw file(s): " + ", ".join(missing_raw)
    )

preparation = prepare_tox21_external_validation(ROOT)
display(pd.DataFrame(preparation["endpoint_summaries"]))
print(
    {
        "joined_rows": preparation["joined_rows"],
        "registry": preparation["registry_path"],
        "manifest": preparation["manifest_path"],
    }
)


,endpoint,rows,observed_labels,positive_count,negative_count,positive_prevalence,standardization_success
0,SR-ARE,647,555,93,462,0.167568,645
1,SR-ATAD5,647,622,38,584,0.061093,645
2,SR-MMP,647,543,60,483,0.110497,645
3,SR-p53,647,616,41,575,0.066558,645


{'joined_rows': 647, 'registry': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project\\data\\external\\external_registry.csv', 'manifest': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project\\data\\external\\tox21_challenge_final\\provenance_manifest.json'}


## 2. Run the locked external evaluation

This step loads the existing calibrated bundles, performs exact molecule and scaffold overlap auditing, generates external predictions, computes endpoint-level metrics, runs profile-controlled bootstrap confidence intervals, and records AD/OOD and abstention-stratified results.


In [3]:
from toxicity_screening.external_validation import run_external_validation

external_results = run_external_validation(
    ROOT,
    profile=PROFILE,
    seed=SEED,
)

audit = external_results["audit"]
predictions = external_results["predictions"]
metrics = external_results["metrics"]
bootstrap = external_results["bootstrap"]
calibration = external_results["calibration"]
stratified = external_results["stratified"]
case_studies = external_results["case_studies"]

print("Independence audit")
display(audit)

print("Primary external metrics")
display(metrics.sort_values(["endpoint", "subset"], ignore_index=True))

print("Bootstrap confidence intervals")
display(bootstrap.sort_values(["endpoint", "subset", "metric"], ignore_index=True))


Independence audit


,dataset_name,endpoint,source,official_url,assay_definition,source_independence_evidence,external_rows,observed_labels,standardization_success,exact_smiles_overlap,inchikey_overlap,exact_molecule_overlap,scaffold_overlap,strict_external_eligible,strict_external_fraction_of_observed,note
0,Tox21_Challenge_Final,SR-ARE,NCATS Tox21 Data Challenge 2014,https://tripod.nih.gov/tox21/challenge/data.jsp,Official final-evaluation binary label for SR-...,Official final-evaluation structures and label...,647,555,645,25,26,26,327,531,0.956757,Exact molecule overlaps are excluded; scaffold...
1,Tox21_Challenge_Final,SR-ATAD5,NCATS Tox21 Data Challenge 2014,https://tripod.nih.gov/tox21/challenge/data.jsp,Official final-evaluation binary label for SR-...,Official final-evaluation structures and label...,647,622,645,25,26,26,327,595,0.956592,Exact molecule overlaps are excluded; scaffold...
2,Tox21_Challenge_Final,SR-MMP,NCATS Tox21 Data Challenge 2014,https://tripod.nih.gov/tox21/challenge/data.jsp,Official final-evaluation binary label for SR-...,Official final-evaluation structures and label...,647,543,645,25,26,26,327,520,0.957643,Exact molecule overlaps are excluded; scaffold...
3,Tox21_Challenge_Final,SR-p53,NCATS Tox21 Data Challenge 2014,https://tripod.nih.gov/tox21/challenge/data.jsp,Official final-evaluation binary label for SR-...,Official final-evaluation structures and label...,647,616,645,25,26,26,327,589,0.956169,Exact molecule overlaps are excluded; scaffold...


Primary external metrics


,dataset_name,endpoint,subset,strict_total,coverage,n,positive_prevalence,threshold,roc_auc,pr_auc,...,brier,ece,nll,recall_at_precision_0.80,precision_at_recall_0.80,tn,fp,fn,tp,abstention_rate
0,Tox21_Challenge_Final,SR-ARE,all_strict_evaluable,531,1.000000,531,0.171375,0.290323,0.702160,0.296454,...,0.133451,0.043149,0.456004,0.000000,0.226629,305,135,37,54,0.344633
1,Tox21_Challenge_Final,SR-ARE,decision_supported,531,0.655367,348,0.137931,0.290323,0.734792,0.290518,...,0.110471,0.038433,0.371468,0.000000,0.211957,270,30,30,18,0.344633
2,Tox21_Challenge_Final,SR-ATAD5,all_strict_evaluable,595,1.000000,595,0.058824,0.230769,0.691429,0.150093,...,0.057538,0.051221,0.403532,0.028571,0.058824,508,52,27,8,0.168067
3,Tox21_Challenge_Final,SR-ATAD5,decision_supported,595,0.831933,495,0.050505,0.230769,0.690043,0.099220,...,0.050839,0.049287,0.371219,0.000000,0.050505,447,23,24,1,0.168067
4,Tox21_Challenge_Final,SR-MMP,all_strict_evaluable,520,1.000000,520,0.103846,0.439252,0.867946,0.357678,...,0.087761,0.065825,0.300820,0.000000,0.299320,366,100,12,42,0.282692
5,Tox21_Challenge_Final,SR-MMP,decision_supported,520,0.717308,373,0.064343,0.439252,0.887476,0.307655,...,0.059116,0.038946,0.233839,0.000000,0.178295,318,31,10,14,0.282692
6,Tox21_Challenge_Final,SR-p53,all_strict_evaluable,589,1.000000,589,0.066214,0.285714,0.740163,0.151868,...,0.064837,0.021811,0.277828,0.000000,0.101449,515,35,29,10,0.198642
7,Tox21_Challenge_Final,SR-p53,decision_supported,589,0.801358,472,0.042373,0.285714,0.719027,0.116321,...,0.044400,0.017146,0.229585,0.000000,0.069388,445,7,18,2,0.198642


Bootstrap confidence intervals


,dataset_name,endpoint,subset,metric,iterations_requested,estimate,lower,upper,valid_bootstraps
0,Tox21_Challenge_Final,SR-ARE,all_strict_evaluable,brier,2000,0.133451,0.113918,0.153611,2000
1,Tox21_Challenge_Final,SR-ARE,all_strict_evaluable,mcc,2000,0.225560,0.137907,0.315951,2000
2,Tox21_Challenge_Final,SR-ARE,all_strict_evaluable,pr_auc,2000,0.296454,0.230317,0.380656,2000
3,Tox21_Challenge_Final,SR-ARE,all_strict_evaluable,roc_auc,2000,0.702160,0.650342,0.755759,2000
4,Tox21_Challenge_Final,SR-ARE,decision_supported,brier,2000,0.110471,0.085590,0.137208,2000
5,Tox21_Challenge_Final,SR-ARE,decision_supported,mcc,2000,0.275000,0.135325,0.409679,2000
6,Tox21_Challenge_Final,SR-ARE,decision_supported,pr_auc,2000,0.290518,0.200767,0.413729,2000
7,Tox21_Challenge_Final,SR-ARE,decision_supported,roc_auc,2000,0.734792,0.663350,0.801567,2000
8,Tox21_Challenge_Final,SR-ATAD5,all_strict_evaluable,brier,2000,0.057538,0.042037,0.074700,2000
9,Tox21_Challenge_Final,SR-ATAD5,all_strict_evaluable,mcc,2000,0.106048,0.002999,0.220422,2000


## 3. Coverage, abstention, and applicability-domain interpretation

Performance on the decision-supported subset must be interpreted together with coverage and abstention. The all-strict-evaluable analysis remains the principal discrimination/calibration assessment.


In [4]:
coverage_summary = (
    predictions.groupby(
        ["endpoint", "applicability_domain", "abstention_status"],
        dropna=False,
    )
    .size()
    .rename("n")
    .reset_index()
)
coverage_summary["endpoint_total"] = coverage_summary.groupby("endpoint")["n"].transform("sum")
coverage_summary["fraction"] = coverage_summary["n"] / coverage_summary["endpoint_total"]

display(coverage_summary)
display(stratified.sort_values(["endpoint", "stratum", "stratum_value"], ignore_index=True))


,endpoint,applicability_domain,abstention_status,n,endpoint_total,fraction
0,SR-ARE,borderline,abstain,45,531,0.084746
1,SR-ARE,borderline,supported,120,531,0.225989
2,SR-ARE,inside,abstain,70,531,0.131827
3,SR-ARE,inside,supported,228,531,0.429379
4,SR-ARE,outside,abstain,68,531,0.128060
5,SR-ATAD5,borderline,abstain,12,595,0.020168
6,SR-ATAD5,borderline,supported,172,595,0.289076
7,SR-ATAD5,inside,abstain,16,595,0.026891
8,SR-ATAD5,inside,supported,323,595,0.542857
9,SR-ATAD5,outside,abstain,72,595,0.121008


,dataset_name,endpoint,subset,strict_total,coverage,n,positive_prevalence,threshold,roc_auc,pr_auc,...,ece,nll,recall_at_precision_0.80,precision_at_recall_0.80,tn,fp,fn,tp,stratum,stratum_value
0,Tox21_Challenge_Final,SR-ARE,abstention_status=abstain,531,0.344633,183,0.234973,0.290323,0.621346,0.333119,...,0.078022,0.616760,0.046512,0.255319,35,105,7,36,abstention_status,abstain
1,Tox21_Challenge_Final,SR-ARE,abstention_status=supported,531,0.655367,348,0.137931,0.290323,0.734792,0.290518,...,0.038433,0.371468,0.000000,0.211957,270,30,30,18,abstention_status,supported
2,Tox21_Challenge_Final,SR-ARE,applicability_domain=borderline,531,0.310734,165,0.187879,0.290323,0.678503,0.292691,...,0.049017,0.450959,0.000000,0.240385,87,47,13,18,applicability_domain,borderline
3,Tox21_Challenge_Final,SR-ARE,applicability_domain=inside,531,0.561205,298,0.144295,0.290323,0.735705,0.303751,...,0.056424,0.388918,0.000000,0.210843,186,69,17,26,applicability_domain,inside
4,Tox21_Challenge_Final,SR-ARE,applicability_domain=outside,531,0.128060,68,0.250000,0.290323,0.614187,0.356609,...,0.086702,0.762240,0.058824,0.285714,32,19,7,10,applicability_domain,outside
5,Tox21_Challenge_Final,SR-ARE,scaffold_status=novel,531,0.404896,215,0.265116,0.290323,0.635021,0.383001,...,0.047077,0.634147,0.070175,0.287425,93,65,21,36,scaffold_status,novel
6,Tox21_Challenge_Final,SR-ARE,scaffold_status=seen,531,0.595104,316,0.107595,0.290323,0.719858,0.205462,...,0.060276,0.334799,0.000000,0.158470,212,70,16,18,scaffold_status,seen
7,Tox21_Challenge_Final,SR-ARE,"similarity_quartile=(0.142, 0.351]",531,0.250471,133,0.255639,0.290323,0.664587,0.377245,...,0.073534,0.652286,0.029412,0.318681,67,32,14,20,similarity_quartile,"(0.142, 0.351]"
8,Tox21_Challenge_Final,SR-ARE,"similarity_quartile=(0.351, 0.474]",531,0.250471,133,0.142857,0.290323,0.613112,0.224298,...,0.076958,0.431992,0.000000,0.168317,73,41,10,9,similarity_quartile,"(0.351, 0.474]"
9,Tox21_Challenge_Final,SR-ARE,"similarity_quartile=(0.474, 0.6]",531,0.250471,133,0.157895,0.290323,0.798895,0.456256,...,0.052139,0.354495,0.142857,0.260274,81,31,6,15,similarity_quartile,"(0.474, 0.6]"


## 4. Reproducible representative molecule case studies

Case studies are selected by deterministic rules covering correct high-confidence predictions, errors, abstained/outside-domain molecules, and novel scaffolds where available.


In [5]:
if case_studies.empty:
    print("No eligible external case studies were selected.")
else:
    display(case_studies)


,sample_id,molecule_id,endpoint,original_smiles,standardized_smiles,label,locked_prediction,predicted_class,calibrated_probability,locked_threshold,uncertainty,nearest_training_similarity,scaffold_novelty,applicability_domain,abstention_status,case_category,interpretation,recommendation
0,NCGC00357011-01,MHXFWEJMQVIWDH-UHFFFAOYSA-N,SR-ARE,NC1=C2C(=O)C3=C(C=CC=C3)C(=O)C2=C(O)C=C1OC1=CC...,Nc1c(Oc2ccccc2)cc(O)c2c1C(=O)c1ccccc1C2=O,1,1,1,0.750000,0.290323,0.108713,0.487179,True,inside,supported,correct_active,"{""ood_reasons"": [""novel_scaffold""]}",High-priority toxicity testing
1,NCGC00357283-01,HUHGPYXAVBJSJV-UHFFFAOYSA-N,SR-ARE,OCCN1CN(CCO)CN(CCO)C1,OCCN1CN(CCO)CN(CCO)C1,0,0,0,0.000000,0.290323,0.018919,0.625000,False,inside,supported,correct_inactive,"{""ood_reasons"": []}",Low predicted concern; experimental confirmati...
2,NCGC00356942-01,TWLMSPNQBKSXOP-UHFFFAOYSA-N,SR-ARE,NC1=C(O)C(Cl)=CC(=C1)[N+]([O-])=O,Nc1cc([N+](=O)[O-])cc(Cl)c1O,0,1,1,0.750000,0.290323,0.161235,0.620690,False,inside,supported,false_positive,"{""ood_reasons"": []}",High-priority toxicity testing
3,NCGC00357275-01,FRASJONUBLZVQX-UHFFFAOYSA-N,SR-ATAD5,O=C1C=CC(=O)C2=C1C=CC=C2,O=C1C=CC(=O)c2ccccc21,1,1,1,0.300000,0.230769,0.054415,0.529412,True,inside,supported,correct_active,"{""ood_reasons"": [""novel_scaffold""]}",Experimental confirmation recommended
4,NCGC00357283-01,HUHGPYXAVBJSJV-UHFFFAOYSA-N,SR-ATAD5,OCCN1CN(CCO)CN(CCO)C1,OCCN1CN(CCO)CN(CCO)C1,0,0,0,0.000000,0.230769,0.003020,0.625000,False,inside,supported,correct_inactive,"{""ood_reasons"": []}",Low predicted concern; experimental confirmati...
5,NCGC00357077-01,ZWKNLRXFUTWSOY-QPJJXVBHSA-N,SR-ATAD5,N#C\C=C\C1=CC=CC=C1,N#C/C=C/c1ccccc1,0,1,1,0.312500,0.230769,0.147927,0.550000,False,inside,supported,false_positive,"{""ood_reasons"": []}",Experimental confirmation recommended
6,NCGC00356942-01,TWLMSPNQBKSXOP-UHFFFAOYSA-N,SR-MMP,NC1=C(O)C(Cl)=CC(=C1)[N+]([O-])=O,Nc1cc([N+](=O)[O-])cc(Cl)c1O,1,1,1,1.000000,0.439252,0.093843,0.645161,False,inside,supported,correct_active,"{""ood_reasons"": []}",High-priority toxicity testing
7,NCGC00357092-01,IRUDSQHLKGNCGF-UHFFFAOYSA-N,SR-MMP,CCCCC(C)=C,C=C(C)CCCC,0,0,0,0.000000,0.439252,0.018931,0.428571,False,borderline,supported,correct_inactive,"{""ood_reasons"": []}",Low predicted concern; experimental confirmati...
8,NCGC00261162-01,VDCDWNDTNSWDFJ-UHFFFAOYSA-N,SR-MMP,OC1=C(O)C(=CC(=C1)[N+]([O-])=O)[N+]([O-])=O,O=[N+]([O-])c1cc(O)c(O)c([N+](=O)[O-])c1,0,1,1,1.000000,0.439252,0.040569,0.655172,False,inside,supported,false_positive,"{""ood_reasons"": []}",High-priority toxicity testing
9,NCGC00261160-01,NIAOXEKTWMHVEO-UHFFFAOYSA-N,SR-p53,[I-].CC[N+]1=C(SC2=C1C=CC=C2)\C=C\C=C1\N(C)C2=...,CC[n+]1c(/C=C/C=C2/N(C)c3ccccc3C2(C)C)sc2ccccc21,1,1,1,0.384615,0.285714,0.140120,0.409091,True,borderline,supported,correct_active,"{""ood_reasons"": [""novel_scaffold""]}",Experimental confirmation recommended


## 5. Render manuscript-ready figures in an isolated plotting process

Plotting is delegated to a fresh Python process with one-thread numerical settings.


In [6]:
from toxicity_screening.external_validation import render_external_validation_figures

import os

os.environ["TOXICITY_PLOT_PYTHON"] = r"D:\Users\anaconda3\python.exe"

from toxicity_screening.external_validation import (
    render_external_validation_figures,
)

figure_paths = render_external_validation_figures(ROOT)

for figure_path in figure_paths:
    print(figure_path.relative_to(ROOT))

figures\external_validation_roc_curves.png
figures\external_validation_pr_curves.png
figures\external_validation_reliability.png
figures\external_validation_similarity_performance.png
figures\external_validation_domain_coverage.png


## Completion gate

The external-validation claim is restricted to the four Tox21 stress-response endpoints. `herg_blockade` and `ames_mutagenicity` remain explicitly unvalidated externally.


In [7]:
required_outputs = [
    ROOT / "data/external/tox21_challenge_final/provenance_manifest.json",
    ROOT / "data/external/external_registry.csv",
    ROOT / "results/external_validation/independence_audit.csv",
    ROOT / "results/external_validation/external_predictions.csv",
    ROOT / "results/external_validation/external_metrics.csv",
    ROOT / "results/external_validation/external_bootstrap_ci.csv",
    ROOT / "results/external_validation/external_calibration.csv",
    ROOT / "results/external_validation/external_ad_stratified_metrics.csv",
    ROOT / "results/external_validation/external_case_studies.csv",
    ROOT / "results/external_validation/status.json",
]

missing_outputs = [str(path.relative_to(ROOT)) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise RuntimeError(
        "External validation did not produce required artifacts: " + ", ".join(missing_outputs)
    )

status = json.loads((ROOT / "results/external_validation/status.json").read_text(encoding="utf-8"))
assert set(status["validated_endpoints"]) == set(EXTERNAL_ENDPOINTS)
assert set(status["not_externally_validated_endpoints"]) == {"herg_blockade", "ames_mutagenicity"}
assert predictions["exact_molecule_overlap"].sum() == 0
assert predictions["strict_external_eligible"].all()

print(status)
print("Notebook 20 external-validation completion gate passed.")


{'created_at': '2026-08-11T17:18:04.287164+00:00', 'status': 'completed', 'profile': 'full', 'seed': 20260723, 'bootstrap_iterations': 2000, 'validated_endpoints': ['SR-ARE', 'SR-ATAD5', 'SR-MMP', 'SR-p53'], 'not_externally_validated_endpoints': ['herg_blockade', 'ames_mutagenicity'], 'strict_external_predictions': 2235, 'claim': 'External validation was performed only for the four Tox21 stress-response endpoints after exact-overlap exclusion.', 'limitations': 'hERG and Ames were not externally validated.'}
Notebook 20 external-validation completion gate passed.
